# Diffusion-Based Framework

In [1]:
# Cell 1: Imports + basic config
import os, glob, random, math, tqdm
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
DATA_DIR = Path.cwd() / "datasets"

# >>> USER: set this to the folder containing your preprocessed .npz files
processed_data_dir = DATA_DIR / "AIST/processed" 

# quick device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# check dataset files
files = sorted(glob.glob(os.path.join(processed_data_dir, "*.npz")))
print("Found .npz files:", len(files))
if len(files) > 0:
    print("Sample file:", files[0])
else:
    print(f"WARNING: No .npz files found in preprocessed_data_dir={processed_data_dir}")


Device: cuda
Found .npz files: 4355
Sample file: d:\STUDIES\MTech\#MTP\codes\vimo\datasets\AIST\processed\gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz


In [2]:
# Cell 2: Inspect structure of one .npz file (sanity check)
if len(files) > 0:
    sample = np.load(files[0])
    print("Keys in npz:", list(sample.keys()))
    for k in sample.files:
        print(k, "->", np.array(sample[k]).shape, np.array(sample[k]).dtype)
    # expected (example): p2d_cond: (views, S, 17, 3)  ; m3d_gt: (S, 151)
else:
    print("No files to inspect.")


Keys in npz: ['p2d_cond', 'm3d_gt']
p2d_cond -> (9, 150, 17, 3) float32
m3d_gt -> (150, 151) float32


In [3]:
# Cell 3: ViMo Dataset class (loads single view randomly if multiple views)
#from collections import defaultdict
#VAL_IDX_MAP = defaultdict(lambda:random.sample(range(N_VIEWS), N_VIEWS - t_views), {})
import random, traceback

class ViMoDataset(Dataset): # dont use num_workers in dataloader, its not working!
    N_VIEWS = 9 # class_vars, total view must be 9 accoding to dataset
    VAL_IDX_MAP = {} # hash map['file'->list('val_idx')] to remember val_idxs
    # acces via class_name to modify & via self to read as safety measure
    """
    Parameters:
        - file_list: list of files from which to load data from
        - train: if enabled, each view of 2d pose is considered as single training data paired with resp 3d motion
        - conf_thresh: set keypoints with low confidence threshold to zero i.e. mark as missing
        - t_views: no.of views to consider for training per file
        - v_views: no.of views to consider for validation per file
    Functions:
        - pose_preprocessor(): fn to stabilze (i.e. normalize) 2d poses that represent image pixel coordinated sys
    """
    def __init__(self, file_list, t_views=8, conf_thresh=0.2, train=True):
        self.file_list = file_list
        self.train = train
        self.conf_thresh = conf_thresh
        if (t_views > self.N_VIEWS and isinstance(t_views, int)):
            raise RuntimeError("Max views per sample is 9! (enter proper integer)")
        if (t_views < self.N_VIEWS and self.VAL_IDX_MAP=={}):
            raise RuntimeError("Val_Idx_Map need to be pre-initialized to make use multiple workers without race condition")
        self.t_views = t_views # no.of views per p2d in training
        self.v_views = self.N_VIEWS - t_views # no.of views per p2d in training
        
    def pose_preprocessor(self, p2d):
        p2d[np.isnan(p2d)] = 0 # reset nan values
        
        # Centering: Subtract the coordinates of a reference joint (pelvis)
        root = p2d[:,:1] # pelvis (joint id=0)
        body = p2d[:,1:] # rest all joints
        body[:,:,:2] = body[:,:,:2] - root[:,:,:2] # (S,16,2) - (S,1,2)

        # Confidence Filtering: Set keypoints with low confidence to zero
        body[body[:,:,2] < self.conf_thresh] = 0
        return np.concatenate((root, body), axis=1)
        
    def __len__(self):
        if self.train:
            return len(self.file_list) * self.t_views
        else:
            return len(self.file_list) * self.v_views

    def __getitem__(self, idx):
        file_idx = idx
        if self.train:
            file_idx = idx // self.t_views
        else:
            file_idx = idx // self.v_views
        
        file_path = self.file_list[file_idx]
        file_name = os.path.basename(file_path)
        data = np.load(file_path)

        view_idx = 0
        if self.train: # training dataset loader    
            view_idx = idx % self.t_views
            if self.t_views < self.N_VIEWS:
                all_idx = list(np.arange(self.N_VIEWS))
                #print("val_idx for this file:", VAL_IDX_MAP[file_name])
                idx_map = [idx for idx in all_idx if idx not in self.VAL_IDX_MAP[file_name]]
                #print(f"remap order: {list(enumerate(idx_map))}")
                view_idx = idx_map[view_idx]

        else:  # validation dataset loader
            view_idx = idx % self.v_views
            view_idx = self.VAL_IDX_MAP[file_name][view_idx]
            
        #print(f"file_name={file_name},  p2d.shape={data['p2d_cond'].shape}, fetching view_idx={view_idx}")
        p2d = data['p2d_cond'][view_idx]  # (S, 17, 3) expected
        m3d = data['m3d_gt']    # (S, 151) expected
        self.pose_preprocessor(p2d)
        if np.isnan(m3d).any():
            raise ValueError(f"3D motions from preprocessed data(file={file_name}) contained nan values!")

        # to tensors: (S,17,3) -> float32 ; (S,151) -> float32
        p2d_t = torch.from_numpy(p2d).float()
        m3d_t = torch.from_numpy(m3d).float()
        return {'m3d': m3d_t, 'p2d': p2d_t, 'view_idx':view_idx, 'file_name': file_name}
    
    @classmethod
    def get_train_val_pair(cls, file_list, t_views=8, conf_thresh=0.2):
        for file_path in file_list: # to avoid resetting multiple times in train_ds and val_ds
            file_name = os.path.basename(file_path)
            ViMoDataset.VAL_IDX_MAP[file_name] = random.sample(range(cls.N_VIEWS), cls.N_VIEWS - t_views)
        return (cls(file_list, t_views, conf_thresh, train=True), cls(file_list, t_views, conf_thresh, train=False))


In [4]:
# quick loader test
t_ds,v_ds = ViMoDataset.get_train_val_pair(files[:4], t_views=7)
dl = DataLoader(t_ds, batch_size=2, shuffle=False, pin_memory=True)

it = iter(dl)
batch_idx = 0
while True:
    try:
        batch = next(it)
    except StopIteration:
        print("DataLoader exhausted. Total batches:", batch_idx)
        break
    except Exception as e:
        print(f"Error fetching batch {batch_idx}. exception={e}")
        traceback.print_exc()
        break

    print(f"Batch {batch_idx}: m3d.shape= {batch['m3d'].shape}, p2d.shape= {batch['p2d'].shape}," , \
        f"view_idx= {batch['view_idx']}, file_name= {batch['file_name']}")
    batch_idx += 1

print(f"VAL_IDX_MAP: {ViMoDataset.VAL_IDX_MAP}")

Batch 0: m3d.shape= torch.Size([2, 150, 151]), p2d.shape= torch.Size([2, 150, 17, 3]), view_idx= tensor([1, 2]), file_name= ['gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz', 'gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz']
Batch 1: m3d.shape= torch.Size([2, 150, 151]), p2d.shape= torch.Size([2, 150, 17, 3]), view_idx= tensor([3, 4]), file_name= ['gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz', 'gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz']
Batch 2: m3d.shape= torch.Size([2, 150, 151]), p2d.shape= torch.Size([2, 150, 17, 3]), view_idx= tensor([6, 7]), file_name= ['gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz', 'gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz']
Batch 3: m3d.shape= torch.Size([2, 150, 151]), p2d.shape= torch.Size([2, 150, 17, 3]), view_idx= tensor([8, 0]), file_name= ['gBR_sBM_cAll_d04_mBR0_ch01_clip[0000-0150].npz', 'gBR_sBM_cAll_d04_mBR0_ch01_clip[0105-0255].npz']
Batch 4: m3d.shape= torch.Size([2, 150, 151]), p2d.shape= torch.Size([2, 150, 17, 3]), view_idx= ten

In [ ]:
# Cell 4: Diffusion schedule + sampling utilities
def make_beta_schedule(steps, start=1e-4, end=0.02):
	"""
	Small betas (0.0001 → 0.02) mean each step adds only a tiny amount of noise
	"""
    # linear schedule (adjust to match paper if needed)
	return torch.linspace(start, end, steps, dtype=torch.float32)

# hyperparams (tweak to match paper)
T = 1000  # total diffusion timesteps (paper config variable)
betas = make_beta_schedule(T).to(device)
alphas = 1.0 - betas # ~appox [1-0]
alphas_cumprod = torch.cumprod(alphas, dim=0)  # \bar{alpha}_t => cummulative product

print(f"beta_1 = {betas[0]:.6f}")
print(f"beta_T = {betas[-1]:.6f}")
print(f"alpha_1 = {alphas[0]:.6f}")
print(f"alpha_T = {alphas[-1]:.6f}")
print(f"\nCRITICAL VALUES:")
print(f"alpha_bar_1 = {alphas_cumprod[0]:.6f}")
print(f"alpha_bar_500 = {alphas_cumprod[499]:.6f}")
print(f"alpha_bar_T = {alphas_cumprod[-1]:.10f}")
print(f"\nSignal retention at T: {alphas_cumprod[-1].item()*100:.8f}%")
print(f"Noise level at T: {(1 - alphas_cumprod[-1]).item()*100:.8f}%")

# Calculate sqrt(alpha_bar) and sqrt(1 - alpha_bar)
sqrt_alpha_bar = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alpha_bar = torch.sqrt(1 - alpha_bars)

plt.figure(figsize=(8, 5))
plt.plot(sqrt_alpha_bar.numpy(), label=r"$\\sqrt{\\bar{\\alpha}_t}$ (signal term)", linewidth=2)
plt.plot(sqrt_one_minus_alpha_bar.numpy(), label=r"$\\sqrt{1-\\bar{\\alpha}_t}$ (noise term)", linewidth=2)
plt.xlabel("Diffusion Step $t", fontweight='bold')
plt.ylabel("Multiplier", fontweight='bold')
plt.title(r"Signal vs Noise in Diffusion Process", fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# helper q_sample: forward noising: x_t = sqrt(alpha_bar) * x0 + sqrt(1-alpha_bar) * noise
def q_sample(x0, t, noise=None):
    """
    x0: (B, S, D)
    t: (B,) long tensor in [0..T-1]
    noise: same shape as x0 or None (then sampled)
    """
    if noise is None:
        noise = torch.randn_like(x0)
    # gather alpha_bar for each batch
    a_bar_batch = alphas_cumprod[t]
    a_bar_scaled = a_bar_batch.view(-1, 1, 1).to(x0.device)  # (B,1,1)
    #-1 means "infer this dimension from the length of the input".
    # The two 1s add singleton dimensions, making it easy to broadcast in later operations.

    xt = torch.sqrt(a_bar_scaled) * x0 + torch.sqrt(1.0 - a_bar_scaled) * noise
    return xt, a_bar_batch

# tiny sanity test with fake data
B = 2
# create a fake x0 shape (B, S, D) from earlier batch if available
if len(files) > 0:
    sample_x0 = batch['m3d'][:B].to(device)  # (B, S, 151)
    t_rand = torch.randint(0, T, (B,), device=device) # (B,) random timesteps
    xt, a_bar = q_sample(sample_x0.to(device), t_rand)
    print(f"q_sample xt shape: {xt.shape}, rand_timesteps={t_rand}, alpha_bar={a_bar} at resp timesteps")
else:
    print("No dataset: skip q_sample test.")

alpha_bar.shape for all timesteps: torch.Size([1000])
q_sample xt shape: torch.Size([2, 150, 151]), rand_timesteps=tensor([675, 842], device='cuda:0'), alpha_bar=tensor([0.0097, 0.0008], device='cuda:0') at resp timesteps


## Concepts Recap
To brush up the concepts underlying behind the code in fundamental aspects..


### Positional Encoder
The formula for the encoding is:
- For even indices: PE(pos, 2i) = sin(pos / (10000^(2i/d_model)))
- For odd indices: PE(pos, 2i+1) = cos(pos / (10000^(2i/d_model)))

Where:
- pos is the position of the token in the sequence.
- i is the specific dimension within the embedding vector.
- d is the dimensionality of the embeddings.
- 10000^2i/d​ ensures that each dimension of the positional embedding has a different frequency.

#### References:
- [geekforgeeks](https://www.geeksforgeeks.org/nlp/positional-encoding-in-transformers/)
- [math behind embeddings](https://medium.com/autonomous-agents/math-behind-positional-embeddings-in-transformer-models-921db18b0c28)

In [6]:
class PositionalEncoding(nn.Module):
    """
    Common Position Encoder model (fixed, i.e. non-learnable) 
    for sequences ([B, 0...S, d] -> [B, 0..S, d]) & timesteps ([B,1]->[B,d])
    Parameters:
        - `d_model` : feature dimension (embedding size).
        - `max_len` : maximum sequence length supported.

    Forward:
        - input `x`: (Batch, S-Frames, C-HiddenLayerChannels)
        - HiddenLayer => projected feature dimention of 2d-pose/3d-motion
        - returns: x + PE
    """
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        pe = torch.zeros(max_len, d_model)

        # arrange => [0..max_len-1], dim=1d-vector (i.e. dim idx=0), unsqueeze(1) => add dim at idx=1. 
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        # so pos.shape = (max_len, 1), i.e. its a column vector
        
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # shape=(d_model/2)
        # denominator (frequency) => 10000^(2*i/d_model), where i is the embedding index
        # pos*div_term => [max_len,1]*[1,d_model], i.e. broadcasting to (max_len, d_model/2)
        
        pe[:, 0::2] = torch.sin(pos * div_term) # Fills even cols (0, 2, 4, ...) of embeddings with sine(pos*freq)
        pe[:, 1::2] = torch.cos(pos * div_term) # Fills odd cols (1, 3, 5, ...) of embeddings with cos(pos*freq)
        self.register_buffer('pe', pe) # Registers as a model.buffer (not a parameter, for faster access)

    def sequential_forward(self, x): # Forward for sequence data: x shape (B,S,C)
        assert x.size(-1) == self.pe.size(1), f"PE_seq: Feature Dimension (d_model:{self.d_model}) mismatch!"
        S = x.size(1)
        assert S <= self.pe.size(0), f"PE_seq: Length of Sequence exceeds max_len({self.max_len}) contraint, increase the param val!"
        return x + self.pe[:S].unsqueeze(0) # shapes: x=(B,S,C) + pe=(1,S,C) -> (B,S,c) how?
        # Adding (B,S,C) + (1,S,C) broadcasts the leading 1 to B, producing (B,S,C). So you do NOT need to manually repeat pe per batch

    def timestep_forward(self, t): # Forward for timesteps: t shape (B,)
        return self.pe[t] # returns (B,d_model)

    def forward(self, x):
        assert x.dim() <= 3, "Position Encoder cant handle more than 3 dimentions, reshape accordingly!"
        if (x.dim() == 3):  # sequence data
            return self.sequential_forward(x)
        else: # timestep data
            if (x.dim() == 2) and (x.size(-1) == 1):
                x = x.squeeze(1)  # make it (B,)  1D list for indexing
            if x.dim() == 1:
                return self.timestep_forward(x)
            else:
                raise AssertionError(f"Position Encoder not implemented for {x.dim()} dimention inputs!")

### Transformer Encoder
- `MultiheadAttention.Forward(...)` => `need_weights (bool)` – If True, returns attn_output_weights in addition to attn_outputs. Set `need_weights=False` to use the optimized scaled_dot_product_attention and achieve the best performance for MHA. Default: True.
-`TransformerEncoderLayer(...)`: 
    1. `dim_feedforward` = `embedding_dim` × `4` => a widely accepted convention? based on original paper ('Attention Is All You Need') design, ensuring the FFN provides a strong nonlinearity and model capacity boost between attention layers.
    2. `batch_first (bool)` = input and output tensors.shape => `(batch, seq, feature)` if True else [Default: by orignal design nature] False => `(seq, batch, feature)`.


#### References
- [implementation from scratch](https://medium.com/data-scientists-diary/implementation-of-transformer-encoder-in-pytorch-daeb33a93f9c)
- [fig. self attention vs cross attention](https://media.licdn.com/dms/image/v2/D5622AQES_5kWgZ8O4Q/feedshare-shrink_800/feedshare-shrink_800/0/1719243502822?e=1762992000&v=beta&t=wi5l_z5wwjGKq2-uD8rnsZtAXRmBVXy-w1Vwc36q0OE)
- [attention implementations](https://medium.com/@heyamit10/implement-self-attention-and-cross-attention-in-pytorch-cfe17ab0b3ee)

---

## ViMo Denoiser FAQ:

1. Why `film_in = p_agg + t_emb` (addition) instead of concat?

> - Both are valid fusion strategies:
>    - Addition: fast, parameter-light, assumes p_agg and t_emb live in the same semantic space; the MLP sees their sum.
>    - Concatenation + MLP: more expressive since the MLP can learn arbitrary interactions between p and t.
>
> - The paper doesn't force either; both are reasonable. If you want higher capacity, a concatenation `film_in = torch.cat([p_agg, t_emb], dim=-1)` followed by an MLP to `2*d_model` is a straightforward modification.

2. Why use `x * (1 + gamma)` instead of `x * gamma`?

> Using 1 + γ is a common trick so that if γ is initialized to zeros, the multiplicative term is initially identity (1 + 0 = 1). This helps optimization stability: the FiLM block starts as a no-op (identity) and learns residual modulation rather than arbitrarily scaling features to zero at the start.
>
> With `x * gamma`, if gamma starts near zero the model will output near-zero features (not identity), which can hurt signal flow. `1 + gamma` ensures the multiplicative path is near identity initially.
>
> This is similar to how residual connections are encouraged to be identity-initialized to aid training.

3. ReLU vs GELU (Gaussian Error Linear Unit) activation fn in any transformer/diffusion frameworks)

> Smoother output and richer gradients: GELU does not abruptly zero out negative values like ReLU but instead allows small negative activations. This results in smoother gradients, better optimization, and reduced chance of "dead" neurons.​
>
> Standard in transformer and diffusion models: Transformer-based networks (like BERT and vision transformers), as well as diffusion models (which ViMo is based on), almost universally use GELU activations for MLPs because experiments have shown they yield better performance and training stability.

4. MLP (LinearBlocks/ feed-fordward network in transformers convention): 
    
    - Why two linear layers instead of one?
    
    > A single linear layer would just apply an affine transform: y=Wx+b. That can only perform one linear mapping — no nonlinearity, no cross-feature interactions beyond re-weighting.
    > 
    > For very large models or high-resolution signals, you could use `Linear(d,4d) → GELU → Linear(4d,d)` and it would behave more like a Transformer FFN.
    > Using two linear layers with a nonlinearity in between makes it a 1-hidden-layer MLP, which can model richer nonlinear functions of the input.
    
    - feed-forward “expand-reduce” pattern (d → 4d → d):
    
    > The Transformer’s FFN block uses d→4d→d because it processes sequence tokens and needs extra capacity to mix information across the feature dimensions of each token.

    - why `time_mlp` => `Linear(d,d) → ReLU → Linear(d,d)`?
    
    > The diffusion timestep is a single scalar embedding, so the network must learn nonlinear ways of mapping different noise levels t into feature space (e.g., scale differently for early vs. late steps).
    >
    > Expanding to 4× width is unnecessary — we’re not trying to mix many feature dimensions, only to remap a single embedding. Thus
    >
    > - Time-embedding MLP: Input= 1 vector per sample, Shape= d→d, why? light nonlinear projection
    > - Transformer FFN: Input= sequence of tokens, Shape= d→4d→d, why? high-capacity mixing per token

5. Why add PE(timestep) to PoseEncoder outputs? (broadcasts across frame axis due to dim mismatch)
> The framework diagram shows arrow(s) from: 
> - `PoseEncoder(p2d) --> Embedding <-- PositionEncoder(T)` and 
> - `Concat[ LinearBlocks(PositionEncoder(T)); Embedding]` to be used as FiLM conidtion input params. That implies the authors intended 't' to influence both the pose encoder and the FiLM MLP. So a faithful implementation is:
>   - Add (Broadcast) `PE(t)` into pose embeddings so the FiLM blocks becomes time-aware.
>   - Also concatenate pooled `time aware pose avg` with `t_emb` (which is LinearBlocks applied over `PE(t)`) to produce FiLM γ/β.

In [7]:
# ViMo-Denoiser FrameWork

class DenoiserBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # fundamental blocks of denoiser
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True) # self-attention for motion tokens
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True) # cross-attention with pose tokens
        self.ff_mlp = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model)) # feed-forward MLPs with GELU act-fn
        # to preserve the pipeline order via adding residual + norm around each major sub-block (self-attn, cross-attn, MLP)
        self.norm_sa = nn.LayerNorm(d_model) # layer norm after self-attn
        self.norm_ca = nn.LayerNorm(d_model) # layer norm after cross-attn
        self.norm_ff = nn.LayerNorm(d_model) # layer norm after ff-mlp

    def forward(self, motion_tokens, pose_tokens, gamma, beta): # film params remains same within single block?
        # because gamma & beta are generated from conditions => encoded(pose), embed(timestep) which are constant per denoising step
        x = motion_tokens # alias overwriting after each block

        # Self-attention (Q:m3d, K:m3d, V:m3d) -> learns its own temporal dependencies
        sa = self.self_attn(query=x, key=x, value=x, need_weights=False)[0]
        x = self.norm_sa(x + sa)
        x = x * (1 + gamma) + beta # apply FiLM after self-attn

        # Cross-attention (Q:m3d, K:p2d, V:p2d) -> conditioned guidance on pose tokens
        ca = self.cross_attn(query=x, key=pose_tokens, value=pose_tokens, need_weights=False)[0]
        x = self.norm_ca(x + ca)
        x = x * (1 + gamma) + beta # apply FiLM after self-attn
        
        # Feed-Forward MLP (finally to introduce non-linearity)
        fm = self.ff_mlp(x)
        x = self.norm_ff(x + fm)
        x = x * (1 + gamma) + beta # apply FiLM after MLP
        return x


class ViMoFrameWork(nn.Module):
    """
    Parameters:
        - motion_dim: flattened dimension of 3D motion input (default 24*6+4+3=151)
        - pose_dim: flattened dimension of 2D pose input (default 17*3=51)
        - max_T: maximum diffusion timesteps (default 1000)
        - embed_dim: Transformer embedding dimension (default 256)
        - n_heads: no. of Attention Heads (parallel subspaces) in Transformer Layers (default 8)
        - m_nlayers: no. of Denoiser Blocks in 3D motion ViMo Framework (default 3)
        - p_nlayers: no. of Transformer Layers in 2D pose encoder (default 2)
        - cond_drop: classifier-free guidance drop probability during training (default 0.25 => 25% unconditioned, 75% conditioned)
        
    Forward:
        - input `x_t`: (B, S, motion_dim) - noisy 3d motion
        - input `timestep`: (B,) long or float on device in [0..T-1]
        - (optional) `p2d`: (B, S, 17, 3) or (B, S, pose_dim) - 2d poses (None for unconditioned)
        - (optional) `drop_off`: probability to drop off the guidance (poses conditioned training)
          unconditioned rate (overrides self.cond_drop if provided), this option for eval-time experiments
        - returns: predicted noise (B, S, motion_dim)
    """
    def __init__(self, motion_dim=151, pose_dim=17*3, max_T=1000, embed_dim=256, n_heads=8, m_nlayers=3, p_nlayers=2, cond_drop=0.25, verbose=False):
        super().__init__()
        assert embed_dim % n_heads == 0, "Embedding dimension must be divisible by the number of heads."
        # save all config for easier access through vars(model) or model.__dict__
        self.motion_dim = motion_dim
        self.pose_dim = pose_dim
        self.max_T = max_T
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.m_nlayers = m_nlayers
        self.p_nlayers = p_nlayers
        self.cond_drop = cond_drop
        self.verbose = verbose
        self.PE = PositionalEncoding(embed_dim, 2*max(motion_dim, pose_dim, max_T)) # shared PE instance

        # motion projections
        self.m3d_features = nn.Linear(motion_dim, embed_dim)

        # pose encoder (temporal aggregator) project per-frame pose => tokens
        self.p2d_features = nn.Linear(pose_dim, embed_dim)  # map flattened per-frame pose to embed_dim
        encoder_layer_p = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*4, batch_first=True) # (B,S,C)
        self.pose_encoder = nn.TransformerEncoder(encoder_layer_p, num_layers=p_nlayers) # default: num_layers=2 (for the pose encoder) why?
        # pose seq are relatively low-dim, so a small encoder captures temporal dependencies while keeping computation lesser.

        # diffusion embed(denoiser_step T)-> use small MLP to produce FiLM params
        self.timestep_embedding = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim))
        
        # stack of denoising modules (each contains self-attn, cross-attn, MLP)
        self.d_modules = nn.ModuleList([DenoiserBlock(embed_dim, n_heads) for _ in range(m_nlayers)])

        # layers to produce FiLM parameters from concatenated[encoded(pose); embeded(timestep)]
        self.film_mlp = nn.Linear(embed_dim*2, embed_dim*2)  # -> gamma/beta flattened
        nn.init.zeros_(self.film_mlp.weight); nn.init.zeros_(self.film_mlp.bias) # to preserve identify intial training

        # output projection back to motion space
        self.out_proj = nn.Linear(embed_dim, motion_dim)

    def forward(self, x_t, timestep, p2d=None, drop_off=None):
        B, S, _ = x_t.shape
        cond_drop = drop_off if drop_off is not None else self.cond_drop
        if self.verbose: print(f"x_t.shape: {x_t.shape}, timestep.shape: {timestep.shape}, p2d.shape: {p2d.shape if p2d is not None else "None"} ...")
        if timestep.dim() == 1:
            timestep = timestep.unsqueeze(-1)  # (B,) -> (B,1) our common PE expects in this form

        # apply classifier-free drop to condition during training
        if p2d is None or cond_drop == 1:
            p_flat = torch.zeros(B, S, self.pose_dim, device=x_t.device)
        else:
            p_flat = p2d.view(B, S, -1) if p2d.dim() > 3 else p2d # flatten per-frame pose if needed
            drop_mask = (torch.rand(B, device=x_t.device) < cond_drop).int().view(B,1,1) # generate random for each batch
            p_flat = p_flat * (1 - drop_mask) # zero-out where dropped, (B,S,pose_dim)*(B,1,1) broadcasts over dim-1,2
            if self.verbose: print(f"drop_mask.shape: {drop_mask.shape},", end=" ")
        if self.verbose: print(f"p_flat.shape: {p_flat.shape},", end=" ")
        
        assert not (torch.isnan(p_flat).any() or torch.isnan(x_t).any() or torch.isnan(timestep).any()), "Model Input should not have nan values!"
        # project pose tokens and encode temporally
        p_embed_raw = self.p2d_features(p_flat)  # (B,S,embed_dim)
        if self.verbose: print(f"p_embed_raw.shape: {p_embed_raw.shape},", end=" ")
        p_embed_pos = self.PE(p_embed_raw)
        if self.verbose: print(f"p_embed_pos.shape: {p_embed_pos.shape},", end=" ")
        p_encoded = self.pose_encoder(p_embed_pos)  # (B, S, d) here batch_first=True
        if self.verbose: print(f"p_encoded.shape: {p_encoded.shape} ...")

        # timestep embedding
        t_pos = self.PE(timestep)  # sinusoidal positional embeddings (B,1) -> (B,embed_dim)
        if self.verbose: print(f"t_pos.shape: {t_pos.shape},", end=" ")
        t_emb = self.timestep_embedding(t_pos)  # (B, embed_dim)
        if self.verbose: print(f"t_emb.shape: {t_emb.shape} ...")
        
        # Add time position encodings to pose tokens
        pt_emb = p_encoded + t_pos.unsqueeze(1)  # (B,S,d) + (B,1,d) => broadcast across frame axis
        if self.verbose: print(f"pt_emb.shape: {pt_emb.shape},", end=" ")
        # Mean pooling: aggregate pose into a single global cond vec representing the whole seq
        p_avg = pt_emb.mean(dim=1)  # (B, sum(embed_dim)/S_frames) i.e. mean over time
        if self.verbose: print(f"p_avg.shape: {p_avg.shape} ...")

        # combine encoded(pose), embedding(timestep) -> FiLM params
        film_in = torch.cat([p_avg, t_emb], dim=-1) # concat on last dim
        if self.verbose: print(f"film_in.shape: {film_in.shape},", end=" ")
        film_params = self.film_mlp(film_in)  # (B, 2*embed_dim)
        if self.verbose: print(f"film_params.shape: {film_params.shape},", end=" ")
        gamma, beta = film_params.chunk(2, dim=-1)  # split 2 chunks across last dim, so each (B, embed_dim)
        # expand (B,1,d) to (B,S,d)
        gamma = gamma.unsqueeze(1).expand(-1, S, -1)
        beta  = beta.unsqueeze(1).expand(-1, S, -1)
        if self.verbose: print(f"gamma.shape: {gamma.shape}, beta.shape: {beta.shape} ...")

        # motion projection + pos enc
        x = self.m3d_features(x_t)  # (B,S,d)
        if self.verbose: print(f"m3d_features.shape: {x.shape},", end=" ")
        x = self.PE(x) 
        if self.verbose: print(f"m3d_PE.shape: {x.shape},", end=" ")
        for denoiser in self.d_modules:
            x = denoiser(x, pt_emb, gamma, beta)  # (S,B,d)
            if self.verbose: print(f"m3d_denoiser.shape: {x.shape},", end=" ")
        
        # finally project back to motion_dim
        out = self.out_proj(x)
        if self.verbose: print(f"m3d_out.shape: {out.shape} ...\n")
        return out

In [8]:
# Tiny model forward test using a batch sample
B = batch['m3d'].shape[0]
model = ViMoFrameWork(motion_dim=batch['m3d'].shape[-1],
                        pose_dim=batch['p2d'].shape[-2]*batch['p2d'].shape[-1],
                        embed_dim=128, n_heads=4, m_nlayers=3, verbose=True).to(device)
model.train()
x0 = batch['m3d'].to(device)
t_rand = torch.randint(0, T, (B,), device=device)
xt, _ = q_sample(x0, t_rand)
p2d = batch['p2d'].to(device)
pred = model(xt, t_rand, p2d)
print(">>> Model output shape:", pred.shape)  # expected (B,S,motion_dim)


x_t.shape: torch.Size([2, 150, 151]), timestep.shape: torch.Size([2]), p2d.shape: torch.Size([2, 150, 17, 3]) ...
drop_mask.shape: torch.Size([2, 1, 1]), p_flat.shape: torch.Size([2, 150, 51]), p_embed_raw.shape: torch.Size([2, 150, 128]), p_embed_pos.shape: torch.Size([2, 150, 128]), p_encoded.shape: torch.Size([2, 150, 128]) ...
t_pos.shape: torch.Size([2, 128]), t_emb.shape: torch.Size([2, 128]) ...
pt_emb.shape: torch.Size([2, 150, 128]), p_avg.shape: torch.Size([2, 128]) ...
film_in.shape: torch.Size([2, 256]), film_params.shape: torch.Size([2, 256]), gamma.shape: torch.Size([2, 150, 128]), beta.shape: torch.Size([2, 150, 128]) ...
m3d_features.shape: torch.Size([2, 150, 128]), m3d_PE.shape: torch.Size([2, 150, 128]), m3d_denoiser.shape: torch.Size([2, 150, 128]), m3d_denoiser.shape: torch.Size([2, 150, 128]), m3d_denoiser.shape: torch.Size([2, 150, 128]), m3d_out.shape: torch.Size([2, 150, 151]) ...

>>> Model output shape: torch.Size([2, 150, 151])


### Losses

1. L_simple => diffusion loss for probabilistic models (origally MSE of episilon). But here we can apply MSE of motions{ground_truth - prediction} as we are predicting the motions directly.<br>
Ref: [Diffusion Loss](https://towardsdatascience.com/diffusion-loss-every-step-explained-8c19c5e1349b/)

2. L_joints => MSE of `FK(R_mat)`{gt - pred} => MSE of world_pos{gt - pred}

3. L_vel (angular velocity) => MSE of `[R_mat(i+1) - R_mat(i)]`{gt - pred}

4. L_foot => MSE of `[FK(R_mat(i+1)) - FK(R_mat(i))]*[fˆi]`{gt - pred}
    - The FK(·) denotes the forward kinematic function converting joint rotations into joint positions. 
    - Note that we use the model’s own prediction fˆi of the binary foot contact label’s following

5. Overall training Loss: `L = L_simple + λ1*L_joints + λ2*L_vel + λ3*L_foot`

In [9]:
from pytorch3d import transforms # contains useful 3D transform utils
from pathlib import Path
import os, re, glob, smplx, torch

SMPL_DIR = DATA_DIR  / "SMPL"
SMPL_MODEL = None

# Foot joint ids for SMPL (refer above)
ANKLE_IDX = (7, 8)   # L_Ankle, R_Ankle
FOOT_IDX  = (10,11)  # L_Foot,  R_Foot
CONTACT_IDX = ANKLE_IDX + FOOT_IDX  # (L_Ankle, R_Ankle, L_Foot, R_Foot) use same order used in preprocessing

# ---------- Try to load SMPL model ----------
def try_load_smpl_model():
    global SMPL_MODEL
    smpl_files = glob.glob(f"{SMPL_DIR}/v*/smpl/*.pkl")

    def version_tuple_from_path(f, iter=2):
        # infer version from parent folder name like "v1.0.0" or "v1.1.0"
        p = Path(f)
        for i in range(iter):
            p = p.parent
        parent = p.name
        nums = re.findall(r"\d+", parent)
        if not nums:
            return (0,)
        return tuple(int(x) for x in nums)

    # Prefer a neutral model if present (case-insensitive); otherwise fall back to first candidate
    neutral_types = [file for file in smpl_files if "neutral" in os.path.basename(file).lower()]
    if len(neutral_types) > 0:
        chosen = max(neutral_types, key=version_tuple_from_path)
    else:
        # fallback: pick highest-version candidate regardless of gender
        chosen = max(smpl_files, key=version_tuple_from_path)
    chosen_pkl = Path(chosen)
    model_folder = chosen_pkl.parent.parent
    print(f"Chosen SMPL model.pkl: {chosen_pkl.name} | folder: {model_folder.name}")

    # Create SMPL model using the neutral gender option
    try:
        SMPL_MODEL = smplx.create(model_path=model_folder, model_type='smpl', gender='neutral', use_pca=False).to(device)
        print(f"SMPL-model: {SMPL_MODEL}, created successfully")
    except Exception as e:
        # If smplx.create fails, re-raise with helpful debug info
        raise RuntimeError(f"smplx.create failed for model_folder={model_folder} with error: {e}")

def fk_for_worldpos(rotmats, root_pos):
    """
    Forward Kinematics (FK): Convert joint rotation matrices and root positions to world joint positions.
    Parameters:
        - rotmats: (B,S,24,3,3) rotation matrices for each joint
        - root_pos: (B,S,3) root joint positions
    returns: j3d_world: (B.S,24,3) world joint positions
    """
    global SMPL_MODEL
    if SMPL_MODEL is None: try_load_smpl_model()
    B, S, _  = root_pos.shape
    root3d = root_pos.reshape(B*S, 3)          # (B*S,3)
    axis3d = transforms.matrix_to_axis_angle(rotmats.reshape(B*S, 24, 3, 3))  # (B*S,24,3)
    # compute joint world positions with SMPL FK
    j3d_world = None
    try:
        # smplx expects global_orient (B,3), body_pose (B,69), transl (B,3)
        # call in batches (smpl can process batch) To feed these into the SMPL model, we have to split the data into:
        # global_orient — rotation of the root joint only (pelvis:joint-id=0)
        # body_pose — rotations of all other 23 joints
        # transl — translation vector for the whole body
        with torch.no_grad():
            go = axis3d[:, 0, :]                        # (_,1,3)=>(_,3)
            body = axis3d[:, 1:, :].reshape(-1, 23*3)   # (_,23,3)=>(_,69)
            tr = root3d                                 # (_,3)
            final_output = []
            for i in range(S): # perform batch-wise to avoid memory bottle neck
                st = i*B; ed = st+B
                output = SMPL_MODEL(global_orient=go[st:ed], body_pose=body[st:ed], transl=tr[st:ed])
                # output.joints shape (S, J, 3) where J >= 24. We'll take the first 24 SMPL joints.
                final_output.append(output.joints[:, :24, :]) # (_,24,3)
            j3d_world = torch.cat(final_output)     
    except Exception as e:
        raise RuntimeError(f"\nSMPL FK failed for input (axis3d:{axis3d.shape}, root3d:{root3d.shape}) -> Error: {e}")

    return j3d_world.reshape(B, S, 24, 3)  # (B,S,24,3)

In [10]:
def compute_losses(m_pred, m_gt, lambda_joints=1.0, lambda_vel=1.0, lambda_foot=1.0):
    """
    Parameters:
        - m_pred, m_gt: (B,S,D) where D = 151 {J3D=24*6 + Foot=4 + Root=3}
        - lambda values for L_joints, L_vel, L_foot resp
    Returns: 
        - Total_Loss(L) = L_simple + λ1*L_joints + λ2*L_vel + λ3*L_foot
          Batchwise reduction using 'mean' as standard, i.e. scalar value
        - Dict containing individual loss (total, simple, joints, vel, foot) per sample, i.e. (B,)
    """
    assert m_pred.shape == m_gt.shape, "Predicted and Ground Truth motion shapes must match!"
    B, S, _ = m_gt.shape
    mse = nn.MSELoss()

    root_pred = m_pred[:,:, -3:]
    root_gt = m_gt[:,:, -3:]     # (B,S,3)
    contacts_pred = m_pred[:,:, -7:-3]
    contacts_gt = m_gt[:,:, -7:-3]     # (B,S,4)
    j6d_pred = m_pred[:, :, :-7].reshape(B, S, 24, -1)
    j6d_gt = m_gt[:, :, :-7].reshape(B, S, 24, -1)     # (B,S,24,6)

    # Convert 6D rotation representation to 3x3 rotation matrices using Gram-Schmidt algorithm
    rmats_pred = transforms.rotation_6d_to_matrix(j6d_pred)  
    rmats_gt = transforms.rotation_6d_to_matrix(j6d_gt)     # (B,S,24,3,3)
    pos3d_pred = fk_for_worldpos(rmats_pred, root_pred)
    pos3d_gt = fk_for_worldpos(rmats_gt, root_gt)       # (B,S,24,3)

    # Diffusion Loss: MSE on motions instead of epsilon/noise
    L_simple = mse(m_pred, m_gt) # (B,S,D) => scalar loss for resp batch

    # Joint Loss: MSE on joint world positions
    L_joints = mse(pos3d_pred, pos3d_gt)  # (B,S,24,3) => scalar loss for resp batch
    
    # Velocity (Angular) Loss: difference between consecutive frames of rotation matrices
    Avel_pred = rmats_pred[:,1:,...] - rmats_pred[:,:-1,...]
    Avel_gt   = rmats_gt[:,1:,...] - rmats_gt[:,:-1,...]
    L_vel = mse(Avel_pred, Avel_gt) # (B,S-1,24,3,3) => scalar loss for resp batch

    # Foot Contact Loss: BCE on foot contact predictions
    vel_pred = pos3d_pred[:,1:,CONTACT_IDX,:] - pos3d_pred[:,:-1,CONTACT_IDX,:]
    # f_bar: (B, S, 4) -> (B, S-1, 4) -> (B, S-1, 4, 1) to multiply with (B, S-1, 4, 3)
    foot_pred = vel_pred*contacts_pred[:, :-1].unsqueeze(-1)
    # L2 norm over all last dims other than (0:B,1:S) => shape: (B, S-1)
    foot_norm = torch.linalg.norm(foot_pred, ord=2, dim=tuple(range(2, foot_pred.dim())))  
    L_foot = torch.mean(foot_norm**2) # apply mean square over S frames, reduction batch via mean => scalar

    total = L_simple + lambda_joints*L_joints + lambda_vel*L_vel + lambda_foot*L_foot
    return total, {'simple': L_simple.item(), 'joints': L_joints.item(), 'vel': L_vel.item(), 'foot': L_foot.item()}

In [11]:
# test loss on dummy
B,S,D = batch['m3d'].shape
pred = torch.randn(B, S, D, device=device)
gt   = batch['m3d'].to(device)
loss_val, loss_dict = compute_losses(pred, gt, lambda_joints=1.0, lambda_vel=1.0, lambda_foot=1.0)
print(f"Batch Loss: {loss_val.item()}, Per Sample Loss Categories: \n{loss_dict}")


Chosen SMPL model.pkl: SMPL_NEUTRAL.pkl | folder: v1.1.0
SMPL-model: SMPL(
  Gender: NEUTRAL
  Number of joints: 24
  Betas: 10
  (vertex_joint_selector): VertexJointSelector()
), created successfully
Batch Loss: 10805.7001953125, Per Sample Loss Categories: 
{'simple': 211.17201232910156, 'joints': 10570.513671875, 'vel': 0.6798343658447266, 'foot': 23.33514404296875}


In [12]:
MODEL_DIR = Path.cwd() / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# optimizer + checkpoint helpers
def make_optimizer(model, lr=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    return opt

def get_model_file_name(model, prefix="", suffix=""):
    return (prefix + f"vimo(m{model.motion_dim}_p{model.pose_dim}_d{model.embed_dim}_h{model.n_heads}" + \
        f"_ml{model.m_nlayers}_pl{model.p_nlayers}_c{round(model.cond_drop*100)})" + suffix)

def get_model_path(model, prefix="", suffix=""):
    return MODEL_DIR / (get_model_file_name(model, prefix, suffix) + "_ckpt.pth")

def save_checkpoint(model, opt, path=None, prefix="", suffix=""):
    if path==None: path=get_model_path(model, prefix, suffix)
    ck = {'model_state': model.state_dict(), 'opt_state': opt.state_dict()}
    torch.save(ck, path)
    print("Saved checkpoint to", path)

def load_checkpoint(model, opt, path=None, prefix="", suffix="", map_location=device):
    if path==None: path=get_model_path(model, prefix, suffix)
    ck = torch.load(path, map_location=map_location)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    print("Loaded checkpoint from", path)
    return path


```py
# quick instantiate optimizer if model exists
model.verbose = False
#print("Model Config:", list(model.__dict__))
opt = make_optimizer(model, lr=1e-5)
print("Optimizer created, no. of Parameters count:", sum(p.numel() for p in model.parameters()))

# Cell 7: 1-2 train-step smoke test + checkpoint save/load
model.train()
opt.zero_grad()
it = iter(dl) # reset

batch = next(it)
B = batch['m3d'].shape[0]
x0 = batch['m3d'].to(device)
t_rand = torch.randint(0, T, (B,), device=device)
xt, noise = q_sample(x0, t_rand)
p2d = batch['p2d'].to(device)

# forward - 1
m0_pred = model(xt, t_rand, p2d)
loss, _ = compute_losses(m0_pred, x0, lambda_joints=0.1, lambda_vel=0.1, lambda_foot=0.1)
print("Loss before backward:", loss.item())

# backward - 1
loss.backward()
opt.step()
print("Performed 1st batch backward + optimizer step.")

batch = next(it)
x0 = batch['m3d'].to(device)
t_rand = torch.randint(0, T, (B,), device=device)
xt, noise = q_sample(x0, t_rand)
p2d = batch['p2d'].to(device)

# forward - 2
m0_pred = model(xt, t_rand, p2d)
loss, _ = compute_losses(m0_pred, x0, lambda_joints=0.1, lambda_vel=0.1, lambda_foot=0.1)
print("Loss before backward:", loss.item())

# backward - 2
loss.backward()
opt.step()
print("Performed 2nd batch backward + optimizer step.")
#print("Loss after backward:", loss.item())

# checkpoint test (save & load)
save_checkpoint(model, opt, suffix="_test")
# load back to a fresh model/opt to verify
new_model = ViMoFrameWork(motion_dim=x0.shape[-1], pose_dim=p2d.shape[-2]*p2d.shape[-1],
                          embed_dim=128, n_heads=4, m_nlayers=3).to(device)
new_opt = make_optimizer(new_model, lr=1e-5)
load_checkpoint(new_model, new_opt, suffix="_test")
```


```py
# Cell 8: Minimal training loop (small warmup run) & validation stub

def validate(model, val_loader, device):
    model.eval()
    avg_loss = 0.0
    sum_loss = 0.0
    count = 0
    with torch.no_grad():
        pbar = tqdm.tqdm(val_loader, desc=f"Validation")
        for b in pbar:
            x0 = b['m3d'].to(device)
            B = x0.shape[0]
            t_rand = torch.randint(0, T, (B,), device=device)
            xt, _ = q_sample(x0, t_rand)
            p2d = b['p2d'].to(device)
            pred = model(xt, t_rand, p2d)
            loss, _ = compute_losses(pred, x0, lambda_joints=0.01, lambda_vel=1, lambda_foot=0.1)
            sum_loss += loss.item() * B
            count += B
            avg_loss = sum_loss / max(1, count)
            pbar.set_postfix(loss=loss.item())
            if (torch.isnan(loss).all().item()):
                print("Loss fully converged to nan, stopping validating !!!")
                break # stop is loss convergence
    model.train()
    return avg_loss

# quick warmup training (only if you have data)
if len(files) > 0:
    sub_files = files[: np.ceil(len(files)/50, casting='unsafe', dtype=int)]
    t_ds, v_ds = ViMoDataset.get_train_val_pair(sub_files)
    print(f"No. of items in dataset: training={len(t_ds)}, validation={len(v_ds)}")

    train_loader = DataLoader(t_ds, batch_size=256, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(v_ds, batch_size=256, shuffle=True, pin_memory=True)

    model = ViMoFrameWork(embed_dim=256, n_heads=8, m_nlayers=3, p_nlayers=2).to(device)
    optimizer = make_optimizer(model, lr=1e-5)
    print(f"Model [Parameters Count:{sum(p.numel() for p in model.parameters())}] => \n{model}")

    epochs = 1  # warmup - increase as needed in full training
    for ep in range(epochs):
        pbar = tqdm.tqdm(train_loader, desc=f"Epoch {ep+1}")
        for batch in pbar:
            optimizer.zero_grad()
            x0 = batch['m3d'].to(device)
            B = x0.shape[0]
            t_rand = torch.randint(0, T, (B,), device=device)
            xt, _ = q_sample(x0, t_rand)
            p2d = batch['p2d'].to(device)
            pred = model(xt, t_rand, p2d)
            loss, lost_dict = compute_losses(pred, x0, lambda_joints=0.01, lambda_vel=1, lambda_foot=0.1)
            pbar.set_postfix(loss=loss.item())
            if (torch.isnan(loss).all().item()):
                print("Loss fully converged to nan, stopping trainning !!!")
                break # stop is loss convergence
            loss.backward()
            optimizer.step()

        # quick validation
        val_loss = validate(model, val_loader, device)
        print("Total validation average loss =", val_loss)

    # final checkpoint
    save_checkpoint(model, optimizer, suffix="_warmup")
else:
    print("Skipping training loop (no data).")
```

```py
# Cell 9: Resume load example (use before long training)

# instantiate model & optimizer with same hyperparams used earlier
resume_model = ViMoFrameWork(motion_dim=batch['m3d'].shape[-1],
                        pose_dim=batch['p2d'].shape[-2]*batch['p2d'].shape[-1],
                        embed_dim=256, n_heads=8, m_nlayers=3).to(device)
resume_opt = make_optimizer(resume_model, lr=1e-5)
load_checkpoint(resume_model, resume_opt, suffix="_warmup")
```

---

### Solving Issues: 

#### loss convergence to nan

```py
np.set_printoptions(suppress=True, precision=6, floatmode='fixed')  # fixed-point, no exponent
single_2dpose = batch['p2d'][0][0].cpu().numpy()  # analyse one frame p2d
print(f"single 2d pose shape:{single_2dpose.shape} :: (mean:{single_2dpose.mean()}, std:{single_2dpose.std()})\n {single_2dpose}")
single_3dpose = batch['m3d'][0][0].cpu().numpy()  # analyse one frame m2d
print(f"single 3d pose shape:{single_3dpose.shape} :: (mean:{single_3dpose.mean()}, std:{single_3dpose.std()})\n {single_3dpose}")
# notice high mean and std in 2d poses
```

```py
for i, p2d in enumerate(batch['p2d']):
    for j, s2dpose in enumerate(p2d):
        if torch.isnan(s2dpose).any().item():
            print(f"checking batch-idx[{i}]: frame-idx[{j}]... nan count:{torch.isnan(s2dpose).sum()} >> single 2d pose shape:{s2dpose.shape} :: (mean:{s2dpose.mean()}, std:{s2dpose.std()})")
```

```py
import pickle
file_name = batch['file_name'][51]
src = file_name.split("_clip", 1)[0]
keypoints_dir = DATA_DIR / "AIST" / "keypoints2d"
p = os.path.join(keypoints_dir, src + ".pkl")
pdata = pickle.load(open(p, "rb"))
p2d_all = pdata.get("keypoints2d", None)
print(f"checking src-file: {file_name} ... ")
print(f">> single 2d pose shape:{p2d_all.shape}, nan-count:{np.isnan(p2d_all).sum()} :: (mean:{p2d_all.mean()}, std:{p2d_all.std()})")
p2d_ds = p2d_all[:, ::2]
p2d_ds[np.isnan(p2d_ds)] = 0
print(f">> single 2d pose shape:{p2d_ds.shape}, nan-count:{np.isnan(p2d_ds).sum()} :: (mean:{p2d_ds.mean()}, std:{p2d_ds.std()})")
```

```py
# Diagnostic A: inspect conditioning path and inputs for NaNs/Infs/large values
@torch.no_grad()
def debug_forward_and_losses(model, batch, device=device):
    model.eval()
    m3d = batch['m3d'].to(device)          # (B,S,D)
    p2d = batch['p2d'].to(device)
    B = m3d.shape[0]
    print("Batch m3d: shape", m3d.shape, "NaNs:", torch.isnan(m3d).sum().item(), "Infs:", torch.isinf(m3d).sum().item(),
          "max abs:", float(m3d.abs().max().item()))
    print("Batch p2d: shape", p2d.shape, "NaNs:", torch.isnan(p2d).sum().item(), "Infs:", torch.isinf(p2d).sum().item(),
          "max abs:", float(p2d.abs().max().item()))
    print()
    
    # Use some random t for sampling check
    t = torch.randint(0, T, (B,), device=device)
    xt, noise = q_sample(m3d, t)

    # run denoiser forward
    out = model(xt, t, p2d)
    print("Model forward output -> shape:", out.shape,
            "| NaN count:", torch.isnan(out).sum().item(),
            "| Inf count:", torch.isinf(out).sum().item(),
            "| max abs:", out.abs().max().item(), "| mean abs:", out.abs().mean().item())

    # prepare inputs same way as model.forward does (best-effort, non-invasive)
    try:
        # replicate model's flattening / CF-drop logic (safe read-only)
        if p2d is None:
            p_flat = torch.zeros(B, batch['p2d'].shape[1], model.pose_dim, device=device)
        else:
            p_flat = p2d.view(B, p2d.shape[1], -1) if p2d.dim() > 3 else p2d
            # note: not applying stochastic drop here for deterministic debug

        print("p_flat stats: mean", float(p_flat.mean().item()), "std", float(p_flat.std().item()),
              "NaNs:", int(torch.isnan(p_flat).sum().item()))

        # pose embeddings
        p_emb_raw = model.p2d_features(p_flat) if hasattr(model, 'p2d_features') else None
        if p_emb_raw is not None:
            print("p_emb_raw: mean", float(p_emb_raw.mean().item()), "std", float(p_emb_raw.std().item()),
                  "NaNs:", int(torch.isnan(p_emb_raw).sum().item()), "max_abs", float(p_emb_raw.abs().max().item()))

        # PE added
        p_emb_pos = model.PE(p_emb_raw) if hasattr(model, 'PE') else p_emb_raw
        print("p_emb_pos: mean", float(p_emb_pos.mean().item()), "std", float(p_emb_pos.std().item()),
              "NaNs:", int(torch.isnan(p_emb_pos).sum().item()), "max_abs", float(p_emb_pos.abs().max().item()))

        # pose encoder output
        p_encoded = model.pose_encoder(p_emb_pos) if hasattr(model, 'pose_encoder') else p_emb_pos
        print("p_encoded: mean", float(p_encoded.mean().item()), "std", float(p_encoded.std().item()),
              "NaNs:", int(torch.isnan(p_encoded).sum().item()), "max_abs", float(p_encoded.abs().max().item()))

        # timestep PE & t_emb
        t_pos = model.PE(t.unsqueeze(-1)) if hasattr(model, 'PE') else None
        t_emb = model.timestep_embedding(t_pos) if (t_pos is not None and hasattr(model, 'timestep_embedding')) else None
        print("t_pos:", None if t_pos is None else (t_pos.shape, "NaNs", int(torch.isnan(t_pos).sum().item()), "maxabs", float(t_pos.abs().max().item())))
        print("t_emb:", None if t_emb is None else (t_emb.shape, "NaNs", int(torch.isnan(t_emb).sum().item()), "maxabs", float(t_emb.abs().max().item())))

        # pt_emb and pooled avg
        pt_emb = (p_encoded + t_pos.unsqueeze(1)) if (t_pos is not None) else p_encoded
        print("pt_emb:", "NaNs", int(torch.isnan(pt_emb).sum().item()), "maxabs", float(pt_emb.abs().max().item()))
        p_avg = pt_emb.mean(dim=1)
        print("p_avg:", "NaNs", int(torch.isnan(p_avg).sum().item()), "maxabs", float(p_avg.abs().max().item()))

        # film_in & film_params
        film_in = torch.cat([p_avg, t_emb], dim=-1) if (t_emb is not None) else p_avg
        print("film_in:", "shape", film_in.shape, "NaNs", int(torch.isnan(film_in).sum().item()), "maxabs", float(film_in.abs().max().item()))

        film_params = model.film_mlp(film_in) if hasattr(model, 'film_mlp') else None
        print("film_params:", None if film_params is None else ("shape", film_params.shape, "NaNs", int(torch.isnan(film_params).sum().item()), "maxabs", float(film_params.abs().max().item())))
    except Exception as e:
        print("Detailed debug exception:", e)

    model.train()
    return out

# run it
dbg_out = debug_forward_and_losses(resume_model, batch)
```

```py
# Diagnostic B: one training step with autograd anomaly detection (captures NaN gradient sources)
model = resume_model
opt.zero_grad()
x0 = batch['m3d'].to(device)
p2d = batch['p2d'].to(device)
B = x0.shape[0]
t = torch.randint(0, T, (B,), device=device)
xt, _ = q_sample(x0, t)

# forward
out = model(xt, t, p2d)
print("Forward done. Any NaNs?:", torch.isnan(out).any().item())

# compute loss (catch exceptions)
try:
    total_loss, ld = compute_losses(out, x0)
    print("Loss computed:", float(total_loss.item()), ld)
except Exception as e:
    print("compute_losses exception:", e)
    raise

# run backward under anomaly detection
try:
    with torch.autograd.detect_anomaly():
        total_loss.backward()
    print("Backward completed (no anomaly).")
except RuntimeError as e:
    print("Autograd detect anomaly raised RuntimeError:", e)
    # print grads nan counts
    nan_grad_params = []
    for n, p in model.named_parameters():
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
            nan_grad_params.append((n, float(p.grad.abs().max().item())))
    print("Params with NaN/Inf grads (count):", len(nan_grad_params))
    for n, maxv in nan_grad_params:
        print("  ", n, maxv)
````

```py
# Diagnostic C: find NaN/Inf in model parameters and optimizer state
def list_nans_in_model_and_optimizer(model, optimizer=None):
    print("=== Model parameter NaN/Inf check ===")
    param_issues = []
    for name, p in model.named_parameters():
        if p is None: 
            continue
        n_nan = int(torch.isnan(p).sum().item())
        n_inf = int(torch.isinf(p).sum().item())
        if n_nan > 0 or n_inf > 0:
            param_issues.append((name, p.shape, n_nan, n_inf, float(p.abs().max().cpu().item())))
    if len(param_issues) == 0:
        print("No NaN/Inf in model.parameters()")
    else:
        for nm, sh, nan_cnt, inf_cnt, maxabs in param_issues:
            print(f"PARAM {nm} shape={sh} NaN={nan_cnt} Inf={inf_cnt} maxabs={maxabs}")

    if optimizer is not None:
        print("\n=== Optimizer state NaN/Inf check ===")
        opt_issues = []
        for k, v in optimizer.state.items():
            # k is parameter, v is dict of state tensors
            for sk, st in v.items():
                if isinstance(st, torch.Tensor):
                    n_nan = int(torch.isnan(st).sum().item())
                    n_inf = int(torch.isinf(st).sum().item())
                    if n_nan > 0 or n_inf > 0:
                        opt_issues.append((k, sk, st.shape, n_nan, n_inf, float(st.abs().max().cpu().item())))
        if len(opt_issues) == 0:
            print("No NaN/Inf in optimizer.state tensors")
        else:
            for pkey, sk, sh, nan_cnt, inf_cnt, maxabs in opt_issues:
                print(f"OPT state for param id {id(pkey)} key={sk} shape={sh} NaN={nan_cnt} Inf={inf_cnt} maxabs={maxabs}")

# Call it
list_nans_in_model_and_optimizer(resume_model, resume_opt)
```

---

#### Final testing (Sequential Denoising)

In [13]:
# Sequential denoising => (deterministic mean) test:
@torch.no_grad()
def sequential_denoise(model, x_T, p2d, T_steps=T, device=device):
    """
    x_T: (B,S,D) initial noisy sample at t=T-1 (we can sample x_T by q_sample(x0, T-1))
    p2d: (B,S,...)
    Returns: x0_hat (B,S,D)
    """
    model.eval()
    x_t = x_T.clone().to(device)
    B = x_t.shape[0]

    for t in reversed(range(0, T_steps)):
        t_tensor = torch.full((B,), t, dtype=torch.long, device=device)
        # model predicts x0_hat (the model was trained to predict m0 directly)
        x0_hat = model(x_t, t_tensor, p2d)
        # compute eps_hat from x_t and x0_hat
        a_bar_t = alphas_cumprod[t].to(device)            # scalar
        sqrt_a_bar = torch.sqrt(a_bar_t)
        sqrt_1_a_bar = torch.sqrt(1.0 - a_bar_t)

        # avoid division by zero at t==0
        denom = sqrt_1_a_bar
        denom = denom if denom != 0 else 1e-8

        eps_hat = (x_t - sqrt_a_bar * x0_hat) / denom

        # compute parameters for mean of q(x_{t-1} | x_t, x0_hat)
        if t > 0:
            a_bar_prev = alphas_cumprod[t-1].to(device)
            # mean formula (paper equation for Gaussian posterior mean)
            coef1 = torch.sqrt(a_bar_prev) * (betas[t] / (1.0 - a_bar_t))
            coef1 = coef1.unsqueeze(0).unsqueeze(1).unsqueeze(2)
            coef2 = torch.sqrt(alphas[t]) * (1.0 - a_bar_prev) / (1.0 - a_bar_t)
            coef2 = coef2.unsqueeze(0).unsqueeze(1).unsqueeze(2)
            mu = coef1 * x0_hat + coef2 * x_t  # broadcast to (B,S,D)
            x_t = mu # deterministic: do not add noise; use mean
        else:
            # t == 0: final x0_hat is our best estimate
            x_t = x0_hat

    model.train()
    return x_t

model.verbose = False
# test sequential denoise: sample x_T from a groundtruth x0
it = iter(dl) # reset
batch = next(it)
x0 = batch['m3d'].to(device)
p2d = batch['p2d'].to(device)
# create x_T by forward-noising to t=T-1 (largest noise)
t_Tminus1 = torch.full((x0.shape[0],), T-1, dtype=torch.long, device=device)
x_T, _ = q_sample(x0, t_Tminus1)
x0_hat = sequential_denoise(model, x_T, p2d)
print("sequential denoise finished; x0_hat shape:", x0_hat.shape)

sequential denoise finished; x0_hat shape: torch.Size([2, 150, 151])


## Final Training

In [ ]:
# Full-scale robust training loop (cell)
import sys, os, time, tqdm, csv
class SuppressPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

def train_fullscale(model, train_loader, val_loader, compute_losses_fn, q_sample_fn, device='cpu', epochs=10, lr=1e-4,
                    ckpt_interval=100, val_interval=25, save_prefix="", skip_invalid_batches=True, loss_csv_file=None):# interval based on total no.of batches trained
    model.to(device) # model path doesn't exist, revert back to default
    optimizer = make_optimizer(model, lr=lr)

    pre_epochs = 0 # read no of loss entries in loss_csv_file
    if loss_csv_file is None:
        loss_csv_file = MODEL_DIR / (get_model_file_name(model, prefix=save_prefix) + "_loss.csv")
    file_exists = os.path.exists(loss_csv_file)

    if file_exists:        
        last_entry = list(csv.DictReader(open(loss_csv_file)))[-1] # skip header auto, read last epoch
        pre_epochs = int(last_entry['epoch'])

        while pre_epochs > 0:
            try:
                load_checkpoint(model, optimizer, prefix=save_prefix, suffix=f"_epoch{pre_epochs}", map_location=device)
                #print(f"Resuming from epoch {pre_epochs} checkpoint.")
                break
            except Exception as e:
                print(f"WARNING: loading ckpt for ep-{pre_epochs} not found: {e}. Trying ep-{pre_epochs - 1}.")
                pre_epochs -= 1 # check for prev models
                continue
            
        if int(last_entry['epoch']) > pre_epochs:
            print(f"Handling loss_csv, rewritting existing data upto {pre_epochs}-epochs only!")
            with open(loss_csv_file, newline='') as f:
                reader = csv.DictReader(f)
                fieldnames = reader.fieldnames
                rows = [r for r in reader if int(r.get('epoch', -1)) <= pre_epochs]
            with open(loss_csv_file, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
        
    total_batchs = 0 # total no of batches procesed
    model.train()
    start_time = time.time()

    for epoch in range(pre_epochs+1, pre_epochs+epochs+1):
        loss = 0; val_loss = 0; metrics={'epoch':epoch, 'step':0}
        pbar = tqdm.tqdm(train_loader, desc=f"n_spl={metrics['step']}, Ep[{epoch}]")

        for step, batch in enumerate(pbar):
            optimizer.zero_grad()
            x0 = batch['m3d'].to(device)
            p2d = batch['p2d'].to(device)
            B = x0.shape[0]
            metrics['step']  = step*train_loader.batch_size + B
            step += 1

            # sample random timesteps for diffusion training (standard)
            t = torch.randint(0, T, (B,), device=device)
            xt, _ = q_sample_fn(x0, t)
            
            try:
                # forward + loss
                with SuppressPrints():
                    out = model(xt, t, p2d)
                # quick forward NaN check
                if torch.isnan(out).any() or torch.isinf(out).any():
                    raise RuntimeError(f"NaN/Inf in model output! stats: NaNs={torch.isnan(out).sum().item()}, Infs={torch.isinf(out).sum().item()}, maxabs={out.abs().max().item()}")

                loss, loss_dict = compute_losses_fn(out, x0) # compute losses
                
                # guard against huge loss
                if not torch.isfinite(loss):
                        raise RuntimeError("Non-finite loss")
                loss.backward() # backward
                
            except Exception as e:                
                # you can run debug cell here or break
                if skip_invalid_batches:
                    print(f"[Step {step}] WARNING: Error raised: {e}; skipping this batch!")
                    continue
                else:
                    raise RuntimeError(f"[Step {step}] Error raised: {e}")
            
            # check for NaN in grads
            grad_nan = any((p.grad is not None) and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                           for p in model.parameters())
            if grad_nan:
                print(f"[Step {step}] WARNING: NaN/Inf gradients detected; skipping optimizer.step() and zero grads.")
                optimizer.zero_grad() # reset
                continue

            optimizer.step()
            total_steps = total_batchs + step

            # loss accumulation
            avg_loss = {'total':float(loss.item())}
            avg_loss.update(loss_dict)
            avg_val_loss = {k:'-' for k,v in avg_loss.items()} # initialize empty

            # periodic validation (random-t eval)
            if (total_steps % val_interval == 0) or step == len(train_loader):
                model.eval()
                # quick val: single batch
                with torch.no_grad():
                    try:
                        vb = next(iter(val_loader))
                    except StopIteration:
                        vb = None
                    if vb is not None:
                        x0v = vb['m3d'].to(device)
                        p2dv = vb['p2d'].to(device)
                        tv = torch.randint(0, T, (x0v.shape[0],), device=device)
                        xtv, _ = q_sample_fn(x0v, tv)
                        predv = model(xtv, tv, p2dv)
                        val_loss, val_loss_dict = compute_losses_fn(predv, x0v)
                model.train()
                # val loss accumulation update
                avg_val_loss['total']   = float(val_loss.item())
                avg_val_loss['simple']  = val_loss_dict['simple']
                avg_val_loss['joints']  = val_loss_dict['joints']
                avg_val_loss['vel']     = val_loss_dict['vel']
                avg_val_loss['foot']    = val_loss_dict['foot'] 
            
            # logging
            pbar.set_description(f"n_spl={metrics['step']}, Ep[{epoch}]")
            pbar.set_postfix_str(f"loss={float(loss.item()):.4f}, L_simple={loss_dict['simple']:.4f}, " + \
                                 f"L_joints={loss_dict['joints']:.4f}, L_vel={loss_dict['vel']:.4f}, L_foot={loss_dict['foot']:.4f}, prev_val_loss={val_loss:.4f}")

            # checkpoint (-1 to disable checkpoint saving)
            if ckpt_interval >= 0 and total_steps % ckpt_interval == 0:
                with SuppressPrints():
                    save_checkpoint(model, optimizer, prefix=save_prefix, suffix=f"ep{epoch}_step{metrics['step']}")

            # save loss metrics
            metrics.update({'L_'+k:v for k,v in avg_loss.items()})
            metrics.update({'VL_'+k:v for k,v in avg_val_loss.items()})
        
            with open(loss_csv_file, mode='a', newline='') as file:
                writer = csv.DictWriter(file, fieldnames=list(metrics.keys()))
                if not file_exists:
                    writer.writeheader() # Write header only if file was empty
                    file_exists = True

                writer.writerow(metrics) # Write the metrics for this epoch

        # epoch end checkpoint
        total_batchs = total_steps
        with SuppressPrints():
            save_checkpoint(model, optimizer, prefix=save_prefix, suffix=f"_epoch{epoch}")

    total_time = time.time() - start_time
    print("Training finished. Total time:", total_time)


In [15]:
# Update the configuration parameters here directly as required  (run training, use small epochs for smoke test)

t_ds, v_ds = ViMoDataset.get_train_val_pair(files, t_views=6, conf_thresh=0.2) # taking 6 views for training, 3 views for validation
train_loader = DataLoader(t_ds, batch_size=128, shuffle=True, pin_memory=True)
val_loader   = DataLoader(v_ds, batch_size=128, shuffle=True, pin_memory=True) # max possible batch_size in my sys

model = ViMoFrameWork(motion_dim=151, pose_dim=17*3, max_T=1000, embed_dim=128, # update vimo model params here (refer framework)
                      n_heads=4, m_nlayers=3, p_nlayers=2, cond_drop=0.25, verbose=False)

loss_fn = lambda m_pred, m_gt: compute_losses(m_pred, m_gt, lambda_joints=0.01, lambda_vel=1, lambda_foot=0.1) # update lambda params for loss here

#print(f"Model [Parameters Count:{sum(p.numel() for p in model.parameters())}] => \n{model}\n")
train_fullscale(model, train_loader, val_loader, loss_fn, q_sample, device=device, epochs=2, lr=5e-5, 
                ckpt_interval=-1, val_interval=5, save_prefix="mini_", skip_invalid_batches=False) # update training params here

Loaded checkpoint from d:\STUDIES\MTech\#MTP\codes\vimo\models\mini_vimo(m151_p51_d128_h4_ml3_pl2_c25)_epoch28_ckpt.pth
Handling loss_csv, rewritting existing data upto 28-epochs only!


n_spl=26130, Ep[30]: 100%|██████████| 205/205 [11:21<00:00,  3.33s/it, loss=3.3234, L_simple=2.1421, L_joints=104.9097, L_vel=0.0387, L_foot=0.9352, prev_val_loss=3.9933] 

Training finished. Total time: 1357.0281171798706


### History (Changes):
- Epochs[1-20] : lr=5e-5, λ_joints=0.01, λ_vel=1, λ_foot=0.1

---
---